# TransformerDecoder

从结构上看, `TransformerDecoder` 和前面的 `TransformerEncoder` 很像: 都是先用 `Embedding` 将离散 token 映射到连续向量空间, 然后堆叠多个由注意力层, 前馈网络, 残差连接, LayerNorm 和 Dropout 组成的 block, 最后用线性层输出 logits.

Encoder 使用的是双向自注意力. 对一个输入序列而言, 注意力分数仍然来自:

$$\frac{QK^T}{\sqrt{d_k}}$$

只要不是 padding, 第 `i` 个 token 理论上可以和任意第 `j` 个 token 建立关系. 也就是说, Encoder 的注意力矩阵是完整可见的 token-to-token 关系矩阵. 它适合做句子分类, 序列理解, 表征抽取这类任务, 因为模型在处理每个位置时可以同时利用左侧和右侧上下文.

Decoder 的核心变化是引入因果约束. 在语言模型里, 第 `i` 个 token 只能根据它之前的 token 预测下一个 token, 不能提前看到未来答案. 所以 Decoder 在 softmax 之前会把注意力分数矩阵的右上三角部分 mask 掉:

$$score_{i,j}=-\infty, \quad j>i$$

这样 softmax 后, 未来位置的权重会变成 0. 第 `i` 行不再代表“从整句所有 token 汇聚信息”, 而是代表“只能从当前位置及其之前的 token 汇聚信息”. 这就是 causal self-attention, 也是自回归生成能够成立的根本原因.

因此, Encoder 和 Decoder 的关键区别可以理解为:

1. Encoder 学习的是完整上下文中的双向关系, 适合理解任务.

2. Decoder 学习的是带时间方向约束的单向关系, 适合生成任务.

3. Encoder 通常用 `attention_mask` 屏蔽 padding; Decoder 除了可能需要屏蔽无效位置, 还必须使用 causal mask 屏蔽未来 token.

本 notebook 中的 Decoder 更准确地说是 GPT 风格的 decoder-only 模型. 它没有原始 Transformer Decoder 中用于读取 Encoder 输出的 cross-attention 层, 只有带因果 mask 的 self-attention. 所以它不是翻译模型里的 encoder-decoder decoder, 而是一个直接用于 next-token prediction 的自回归语言模型.

了更直观的普通 padding batch. 一个 batch 保持 `(batch, seq_len)` 形状, 短序列在右侧补 `<pad>`, 训练时用 `attention_mask` 屏蔽 padding token. 因此当前 attention 中实际有两层约束: 一层是不能看未来的 causal mask, 一层是不能看 padding 的 `attention_mask`.


In [ ]:
import torch
from torch import nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d_model, d_model)

    def _rotate_half(self, x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)
    
    def _rope(self, q, k, base=10000):
        seq_len = q.size(-2)
        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** -freqs
        angles = torch.arange(seq_len, device=q.device)[:, None] * freqs[None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[None, None]
        cos = angles.cos().repeat_interleave(2, dim=-1)[None, None]
        return q * cos + self._rotate_half(q) * sin, \
              k * cos + self._rotate_half(k) * sin

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape

        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self._rope(Q, K)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5

        causal_mask = torch.triu( # 加入 causal mask, 屏蔽右上三角
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )
        mask_value = torch.finfo(score.dtype).min
        score = score.masked_fill(causal_mask[None, None, :, :], mask_value)

        if attention_mask is not None:
            key_mask = attention_mask[:, None, None, :] == 0
            score = score.masked_fill(key_mask, mask_value)

        attn_w = torch.softmax(score, dim=-1)
        output = self.dropout(attn_w) @ V
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        output = self.out(output)
        return output


class TransformerDecoderBlock(nn.Module):
    def __init__(self, ffn_dim, d_model, num_heads, dropout):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attention_mask=None):
        x_norm = self.norm1(x)
        x = x + self.dropout(self.attn(x_norm, attention_mask=attention_mask))
        x_norm = self.norm2(x)
        x = x + self.dropout(self.ffn(x_norm))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers, dropout=0.1, padding_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(ffn_dim, d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, output_dim)

    def forward(self, x, attention_mask=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask)
        return self.out(x)

In [32]:
from datasets import load_dataset

batch_size = 64
subset_size = 30_000
max_length = 256

dataset = load_dataset("larryvrh/Chinese-Poems")
subset = dataset["train"].select(range(min(subset_size, len(dataset["train"]))))

def format_poem(poem):
    # 按照 标题 作者 朝代 内容 拼接成语言模型要学习的一整段文本.
    return {"text": f"{poem["title"]} {poem["author"]} {poem["dynasty"]}\n{poem["content"]}"}

text_dataset = subset.map(format_poem, desc="Formatting poems")
texts = text_dataset["text"]
text_lengths = [len(text) for text in texts]
sorted_lengths = sorted(text_lengths)
unique_chars = sorted(set("".join(texts)))

print(f"样本数: {len(texts)}")
print(f"最大文本长度: {max(text_lengths)}")
print(f"最小文本长度: {min(text_lengths)}")
print(f"字符表大小(不含特殊token): {len(unique_chars)}")
print(f"超过 max_length={max_length} 的样本比例: {sum(length + 1 > max_length for length in text_lengths) / len(text_lengths):.2%}")

样本数: 30000
最大文本长度: 16513
最小文本长度: 13
字符表大小(不含特殊token): 9252
超过 max_length=256 的样本比例: 2.13%


In [35]:
class CharTokenizer:
    def __init__(self, chars):
        self.id_to_token = ["<pad>", "<eos>", "<unk>"] + list(chars)
        self.id_to_text = ["", "", ""] + list(chars)
        self.token_to_id = {token: idx for idx, token in enumerate(self.id_to_token)}
        self.pad_token_id = 0
        self.eos_token_id = 1
        self.unk_token_id = 2
        self.vocab_size = len(self.id_to_token)

    def __call__(self, texts, max_length):
        return {"input_ids": [self.encode(text, max_length) for text in texts]}

    def encode(self, text, max_length):
        ids = [self.token_to_id.get(char, self.unk_token_id) for char in text[:max_length - 1]]
        return ids + [self.eos_token_id]

    def decode(self, ids):
        return "".join(self.id_to_text[int(idx)] for idx in ids)

tokenizer = CharTokenizer(unique_chars)
print(f"vocab_size: {tokenizer.vocab_size}")

vocab_size: 9255


In [ ]:
import torch
from torch.utils.data import DataLoader

def tokenize_fn(examples):
    return tokenizer(examples["text"], max_length)

tokenized_dataset = text_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=text_dataset.column_names,
    desc="Tokenizing by character"
)

def collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros_like(input_ids)

    for i, item in enumerate(batch):
        ids = torch.tensor(item["input_ids"], dtype=torch.long)
        input_ids[i, :len(ids)] = ids
        attention_mask[i, :len(ids)] = 1

    return {
        "input_ids": input_ids,
        "labels": input_ids.clone(),
        "attention_mask": attention_mask,
    }

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True,
    num_workers=0
)

first_batch = next(iter(train_dataloader))
print({key: tuple(value.shape) for key, value in first_batch.items()})
print(f"train batches: {len(train_dataloader)}")

{'input_ids': (64, 256), 'labels': (64, 256), 'attention_mask': (64, 256)}
train batches: 469


In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 1
vocab_size = tokenizer.vocab_size
model = TransformerDecoder(
    vocab_size=vocab_size,
    output_dim=vocab_size,
    ffn_dim=256,
    d_model=128,
    num_heads=8,
    num_layers=6,
    dropout=0.1,
    padding_idx=tokenizer.pad_token_id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

for i in range(epochs):
    for idx, data in enumerate(train_dataloader):
        input_ids = data["input_ids"].to(device)
        labels = data["labels"].to(device)
        attention_mask = data["attention_mask"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(
            logits[:, :-1, :].reshape(-1, logits.size(-1)),
            labels[:, 1:].reshape(-1),
        )

        loss.backward()
        optimizer.step()

        if (idx+1) % 10 == 0:
            print(f"Epoch {i+1}, Step {idx+1}, Loss: {loss.item():.4f}, PPL: {torch.exp(loss).item():.4f}")

Epoch 1, Step 10, Loss: 7.2337, PPL: 1385.3048
Epoch 1, Step 20, Loss: 6.6339, PPL: 760.4633
Epoch 1, Step 30, Loss: 6.5342, PPL: 688.2926
Epoch 1, Step 40, Loss: 6.3934, PPL: 597.8872
Epoch 1, Step 50, Loss: 6.2233, PPL: 504.3584
Epoch 1, Step 60, Loss: 6.1171, PPL: 453.5424
Epoch 1, Step 70, Loss: 5.9588, PPL: 387.1413
Epoch 1, Step 80, Loss: 5.9382, PPL: 379.2537
Epoch 1, Step 90, Loss: 5.9782, PPL: 394.7134
Epoch 1, Step 100, Loss: 5.9397, PPL: 379.8192
Epoch 1, Step 110, Loss: 5.7970, PPL: 329.3213
Epoch 1, Step 120, Loss: 5.7806, PPL: 323.9396
Epoch 1, Step 130, Loss: 5.7787, PPL: 323.3509
Epoch 1, Step 140, Loss: 5.7466, PPL: 313.1301
Epoch 1, Step 150, Loss: 5.6870, PPL: 295.0009
Epoch 1, Step 160, Loss: 5.8028, PPL: 331.2400
Epoch 1, Step 170, Loss: 5.6816, PPL: 293.4138
Epoch 1, Step 180, Loss: 5.6090, PPL: 272.8713
Epoch 1, Step 190, Loss: 5.6415, PPL: 281.8730
Epoch 1, Step 200, Loss: 5.6484, PPL: 283.8438
Epoch 1, Step 210, Loss: 5.6006, PPL: 270.5885
Epoch 1, Step 220, Lo

In [68]:

@torch.no_grad()
def generate(model, tokenizer, prompt, max_length, temperature, top_k, device="cuda"):
    model.eval()
    
    input_ids = torch.tensor([tokenizer.encode(prompt, max_length=len(prompt) + 1)[:-1]], dtype=torch.long, device=device)
    
    for _ in range(max_length):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature
        
        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            next_logits = torch.full_like(next_logits, float('-inf'))
            next_logits.scatter_(1, indices, values)
        
        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        input_ids = torch.cat([input_ids, next_token], dim=1)
        
        if next_token.item() == tokenizer.eos_token_id:
            break
    
    generated_text = tokenizer.decode(input_ids[0])
    return generated_text

model.eval()
prompt = "浣溪沙 李白 唐代\n"
story = generate(model, tokenizer, prompt, max_length, temperature=0.9, top_k=10, device=device)
print(story)


浣溪沙 李白 唐代
江山自有时节，不知不识来。
风来未尽，不尽知来无。

何时时时，天地自有心。
一笑无心处，无人不能留。
